# 1. Importing libraries

In [22]:
import re
import pandas as pd
import numpy as np
from collections import Counter

import plotly.express as px

In [23]:
DATA = 'data/'
EDA = 'eda/'

# 2. Loading data

In [24]:
train = pd.read_csv(f'{DATA}/train.csv')
train

,id,prompt_id,text,model
0,0018b3765972,p081,"When a language dies, a particular astronomy d...",qwen3.8-max
1,0041ac86ee93,p073,Data science communities matter because they a...,claude-sonnet-5
2,005a19083eae,p120,"In the final twilight of time, the last embers...",gemini-3.6-flash
3,009319596f10,p073,Data science communities matter because they t...,gpt-5.6-terra
4,00d3e8c92596,p097,"Deep inside each ear, three tiny loops of bone...",glm-5.3-flash
...,...,...,...,...
2694,ff4344ceda23,p075,"Across centuries of silence and shadow, an inv...",gemini-3.8-flash
2695,ff507658ba82,p092,Though both instruments breathe the exact same...,gemini-3.7-flash
2696,ff50c252810c,p113,A single photon leaves the star as neither wav...,claude-opus-4-8
2697,ffcf6321f7e9,p073,Data science communities matter because they t...,glm-5.3


In [25]:
print(train['prompt_id'].nunique())

150


In [26]:
print(train['model'].value_counts())

model
qwen3.8-max               150
claude-sonnet-5           150
glm-5.3                   150
gpt-6-astra               150
gpt-5.5                   150
kimi-k3                   150
claude-opus-5             150
gemini-3.7-flash          150
claude-opus-4-8           150
gpt-5.6-luna              150
claude-opus-4-6           150
gemini-3.8-flash          150
gemini-3.1-pro-preview    150
glm-5.3-flash             150
gpt-5.6-terra             150
gemini-3.6-flash          150
gpt-5.6-sol               150
claude-fable-5            149
Name: count, dtype: int64


# 3. EDA

## 3.1 Sentence lengths

for each prompt and model response, cut each passage into sentences at the space after a punctuation (.!?) that signals the end of a sentence, and then count the number of words in each sentence. not looking at passage length, because it was randomized by the competition. 

find the average length of each sentence, the standard deviation of the sentence length, average number of sentences, max and min length of sentences, and also the max and min number of sentences 

In [27]:
train['sentence_len'] = train.text.apply(
    lambda t: [len(s.split()) for s in re.split(r'(?<=[.!?])\s+', t) if s])

# one row per model, one column per prompt_id, each cell = list of sentence len
sentence_len_df = train.pivot(index='model', columns='prompt_id', values='sentence_len')

# stats over every sentence each model wrote
per_sentence = (train.explode('sentence_len')
                     .astype({'sentence_len': float})
                     .groupby('model').sentence_len)

sentence_len_df['avg_sentence_len'] = per_sentence.mean().round(1)
sentence_len_df['std_sentence_len'] = per_sentence.std().round(1)

prompt_cols = [c for c in sentence_len_df.columns if c.startswith('p')]

counts = (sentence_len_df[prompt_cols]
          .map(lambda x: len(x) if isinstance(x, list) else np.nan)
          .add_suffix('_count'))

order = [col for c in prompt_cols for col in (c, f'{c}_count')]
sentence_len_df = pd.concat([sentence_len_df, counts], axis=1)[order + ['avg_sentence_len', 'std_sentence_len']]
count_cols = [c for c in sentence_len_df.columns if c.endswith('_count')]

# words per sentence: longest and shortest single sentence each model wrote
sentence_len_df['max_sentence_len'] = per_sentence.max()
sentence_len_df['min_sentence_len'] = per_sentence.min()

# sentences per passage: most and fewest sentences in one passage
sentence_len_df['avg_sentence_count'] = sentence_len_df[count_cols].mean(axis=1).round(1)
sentence_len_df['max_sentence_count'] = sentence_len_df[count_cols].max(axis=1)
sentence_len_df['min_sentence_count'] = sentence_len_df[count_cols].min(axis=1)

sentence_len_df.to_csv(f"{EDA}/sentence_len_df.csv", index=False)
sentence_len_df

prompt_id,p000,p000_count,p001,p001_count,p002,p002_count,p003,p003_count,p004,p004_count,...,p148_count,p149,p149_count,avg_sentence_len,std_sentence_len,max_sentence_len,min_sentence_len,avg_sentence_count,max_sentence_count,min_sentence_count
model,,,,,,,,,,,,,,,,,,,,,
claude-fable-5,"[22, 35, 6, 37]",4,"[19, 29]",2,NaN,NaN,"[30, 40, 30, 27, 32, 39, 27, 27]",8,"[22, 6, 22, 45, 24, 20]",6,...,8,"[13, 24, 24, 23, 22]",5,23.9,10.8,137.0,1.0,4.7,11.0,1.0
claude-opus-4-6,"[40, 47, 30, 29, 37, 80]",6,"[18, 37, 11, 28]",4,"[21, 23, 24, 11, 41]",5.0,"[22, 28, 17, 21]",4,"[14, 22, 28, 44, 20, 23]",6,...,3,"[47, 28, 25, 30, 40, 30, 43, 8, 12]",9,26.9,11.8,93.0,1.0,4.3,9.0,1.0
claude-opus-4-8,"[37, 4, 34]",3,"[31, 20]",2,"[29, 32, 20, 16, 21, 37]",6.0,"[47, 34, 27, 35, 28, 20, 36]",7,"[19, 48, 11, 17, 18]",5,...,5,"[21, 21, 20, 17]",4,24.4,9.9,79.0,3.0,5.1,11.0,2.0
claude-opus-5,"[23, 5, 16]",3,"[37, 8, 53, 28, 11, 61, 12, 38]",8,"[16, 8, 16]",3.0,"[10, 16, 28, 20, 5, 12]",6,"[24, 20, 21, 23, 18, 15, 12]",7,...,2,"[23, 21]",2,23.9,12.7,110.0,1.0,5.2,10.0,1.0
claude-sonnet-5,"[29, 41]",2,"[20, 23, 15, 34, 36]",5,"[25, 22, 25, 20, 30]",5.0,"[32, 22, 23, 18, 24, 41]",6,"[11, 41, 21]",3,...,2,"[27, 40, 44, 38, 26]",5,26.4,10.0,96.0,4.0,4.6,10.0,1.0
gemini-3.1-pro-preview,"[21, 20, 34]",3,"[23, 16, 23, 20, 20]",5,"[20, 28, 19, 25, 24]",5.0,"[19, 35, 19, 24, 21, 9, 13, 10, 16, 18, 39, 10]",12,"[12, 12, 17, 19]",4,...,5,"[31, 20, 16, 16, 22, 14, 17, 25]",8,20.7,7.3,50.0,2.0,5.8,12.0,2.0
gemini-3.6-flash,"[28, 27, 27, 29]",4,"[20, 30, 21, 30]",4,"[25, 20, 23, 21]",4.0,"[31, 21, 13, 24]",4,"[13, 17, 12]",3,...,2,"[28, 31, 13, 37]",4,24.2,10.6,220.0,6.0,4.6,9.0,1.0
gemini-3.7-flash,"[30, 44, 30, 33, 41]",5,"[16, 16, 23]",3,"[34, 27, 36, 35, 25, 22, 30, 34]",8.0,"[23, 26, 29, 17, 28, 25, 17, 22, 26]",9,"[21, 6, 18]",3,...,4,"[24, 22, 30, 18, 22, 32]",6,24.4,8.0,96.0,4.0,4.8,9.0,1.0
gemini-3.8-flash,"[22, 24]",2,"[12, 37]",2,"[25, 27, 25, 32]",4.0,"[22, 15, 24]",3,"[30, 15, 64, 20, 27, 13, 35]",7,...,3,"[13, 14, 15, 29]",4,22.8,9.0,140.0,3.0,4.4,10.0,1.0


In [28]:
FAMILY = {
    'claude-fable-5': 'Claude', 'claude-opus-4-6': 'Claude', 'claude-opus-4-8': 'Claude',
    'claude-opus-5': 'Claude', 'claude-sonnet-5': 'Claude',
    'gemini-3.1-pro-preview': 'Gemini', 'gemini-3.6-flash': 'Gemini',
    'gemini-3.7-flash': 'Gemini', 'gemini-3.8-flash': 'Gemini',
    'gpt-5.5': 'GPT', 'gpt-5.6-luna': 'GPT', 'gpt-5.6-sol': 'GPT',
    'gpt-5.6-terra': 'GPT', 'gpt-6-astra': 'GPT',
    'glm-5.3': 'GLM', 'glm-5.3-flash': 'GLM',
    'kimi-k3': 'Kimi', 'qwen3.8-max': 'Qwen',
}

COLORS = {'Claude': '#2a78d6', 'Gemini': '#eb6834', 'GPT': '#1baf7a',
          'GLM': '#eda100', 'Kimi': '#e87ba4', 'Qwen': '#008300'}

In [30]:
df = sentence_len_df[['avg_sentence_len']].reset_index()
df['family'] = df['model'].map(FAMILY)

fig = px.bar(df, x='avg_sentence_len', y='model', color='family', orientation='h',
             color_discrete_map=COLORS, text_auto='.1f',
             title='Average number of words per sentence for each model',)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=550)
fig.show()

In [ ]:
df = sentence_len_df[['std_sentence_len']].reset_index()
df['family'] = df['model'].map(FAMILY)

fig = px.bar(df, x='std_sentence_len', y='model', color='family', orientation='h',
             color_discrete_map=COLORS, text_auto='.1f',
             title='How much sentence length varies within each model',
             labels={'std_sentence_len': 'Standard deviation of words per sentence (higher = more mixed)',
                     'model': ''})
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=550)
fig.show()

In [32]:
df = sentence_len_df[['max_sentence_len']].reset_index()
df['family'] = df['model'].map(FAMILY)

fig = px.bar(df, x='max_sentence_len', y='model', color='family', orientation='h',
             color_discrete_map=COLORS, text_auto='.0f',
             title='Longest single sentence each model wrote',
             labels={'max_sentence_len': 'Words in the longest sentence', 'model': ''})
fig.add_vline(x=50, line_dash='dash', line_color='grey', annotation_text='50 words')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=550)
fig.show()

In [36]:
df = sentence_len_df[['min_sentence_len']].reset_index()
df['family'] = df['model'].map(FAMILY)

fig = px.bar(df, x='min_sentence_len', y='model', color='family', orientation='h',
             color_discrete_map=COLORS, text_auto='.0f',
             title='Shortest single sentence each model wrote',
             labels={'min_sentence_len': 'Words in the shortest sentence', 'model': ''})
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=550)
fig.show()

In [12]:
n_sents = train.sentence_len.str.len()
df = (n_sents == 1).groupby(train.model).sum().rename('one_sentence').reset_index()
df['family'] = df['model'].map(FAMILY)

fig = px.bar(df, x='one_sentence', y='model', color='family', orientation='h',
             color_discrete_map=COLORS, text_auto=True,
             title='Passages written as a single sentence',
             labels={'one_sentence': 'Number of passages (out of ~150)', 'model': ''})
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=550)
fig.show()

In [37]:
df = n_sents.groupby(train.prompt_id).mean().rename('avg_sentence_count').reset_index()
df['highlight'] = df['prompt_id'].eq('p076').map({True: 'p076', False: 'other prompts'})

fig = px.bar(df, x='prompt_id', y='avg_sentence_count', color='highlight',
             color_discrete_map={'p076': '#2a78d6', 'other prompts': '#c3c2b7'},
             title='Average number of sentences per passage, by prompt',
             labels={'avg_sentence_count': 'Sentences per passage', 'highlight': ''})
fig.add_hline(y=df['avg_sentence_count'].median(), line_dash='dash', line_color='grey',
              annotation_text=f"median {df['avg_sentence_count'].median():.1f}")
fig.show()

data shows that the average sentence length and average number of sentences differs only a little between models so that may be a weak signal.

these models showed more even, steady sentences:
- `gemini-3.1-pro-preview` (7.3)
- `gpt-6-astra` (7.3)
- `qwen3.8-max` (7.4)
- `gpt-5.6-luna` (8.0)
- `gemini-3.7-flash` (8.0) 

while these models showed a mix of very long and short sentences:
- `claude-opus-5` (12.7)
- `claude-opus-4-6` (11.8) 
- `kimi-k3` (11.4)
- `glm-5.3` (11.0)

three models never wrote a sentence longer than ~50 words, even when other models wrote 100–220-word sentences:
- `gpt-6-astra` (49)
- `gemini-3.1-pro-preview` (50)
- `qwen3.8-max` (51)

next lowest are `gpt-5.6-luna` (68) and `claude-opus-4-8` (79). every other model has at least one sentence of 93+ words

one oddity is that for prompt_id = `p076`, most of the models wrote fewer sentences. there is a chance that the prompt asked for only one sentence

`gemini-3.1-pro-preview`, `gpt-6-astra` and `claude-opus-4-8` never wrote a passage that was just one sentence (min = 2). `gpt-5.5` did it most often (8 passages), then `kimi-k3` (6). this ties in with the `p076` oddity: some models go along with a "one long sentence" style and some always break it up.

shortest sentence is not very useful, as most models have a 1-3 word sentence somewhere, while max number of sentences showed no clear pattern

## 3.2 Punctuation

for each prompt and model response, count the number of times each punctuation mark was being used 

In [15]:
PUNCT = {
    'comma': ',', 'period': '.', 'semicolon': ';', 'colon': ':',
    'em_dash': '—', 'spaced_em_dash': ' — ', 'en_dash': '–', 'hyphen': '-',
    'curly_apos': '’', 'straight_apos': "'", 'curly_quote': '“', 'straight_quote': '"',
    'paren': '(', 'exclaim': '!', 'question': '?',
}

# per passage: counts of each mark
counts = pd.DataFrame({name: train.text.str.count(re.escape(mark))
                       for name, mark in PUNCT.items()})

# each cell = dict of the marks that passage used, e.g. {'comma': 5, 'em_dash': 1}
train['punct'] = counts.apply(lambda r: {k: v for k, v in r.items() if v > 0}, axis=1)
punct = train.pivot(index='model', columns='prompt_id', values='punct')

# summary: average rate per 100 words for each mark
rates = counts.div(train.text.str.split().str.len(), axis=0) * 100
punct = punct.join(rates.groupby(train.model).mean().round(2).add_prefix('per100_'))

punct.to_csv("punctuation.csv", index=False)
punct

,p000,p001,p002,p003,p004,p005,p006,p007,p008,p009,...,per100_spaced_em_dash,per100_en_dash,per100_hyphen,per100_curly_apos,per100_straight_apos,per100_curly_quote,per100_straight_quote,per100_paren,per100_exclaim,per100_question
model,,,,,,,,,,,,,,,,,,,,,
claude-fable-5,"{'comma': 10, 'period': 4, 'em_dash': 1, 'hyph...","{'comma': 6, 'period': 3, 'colon': 1, 'em_dash...",NaN,"{'comma': 19, 'period': 8, 'semicolon': 2, 'co...","{'comma': 8, 'period': 7, 'semicolon': 1, 'col...","{'comma': 7, 'period': 3, 'colon': 1, 'straigh...","{'comma': 10, 'period': 6, 'semicolon': 1, 'co...","{'comma': 9, 'period': 2, 'em_dash': 2, 'hyphe...","{'comma': 3, 'period': 1, 'semicolon': 2, 'col...","{'comma': 8, 'period': 5, 'colon': 1, 'em_dash...",...,0.00,0.00,0.60,0.00,1.04,0.00,0.15,0.04,0.00,0.04
claude-opus-4-6,"{'comma': 20, 'period': 6, 'colon': 1, 'em_das...","{'comma': 10, 'period': 5, 'semicolon': 1, 'em...","{'comma': 9, 'period': 5, 'em_dash': 1, 'hyphe...","{'comma': 6, 'period': 4, 'colon': 1, 'straigh...","{'comma': 9, 'period': 6, 'colon': 1, 'em_dash...","{'comma': 6, 'period': 4, 'em_dash': 1, 'space...","{'comma': 10, 'period': 7, 'semicolon': 1, 'co...","{'comma': 6, 'period': 2, 'em_dash': 1}","{'comma': 11, 'period': 5, 'hyphen': 1}","{'comma': 5, 'period': 4, 'em_dash': 1, 'hyphe...",...,0.37,0.00,0.77,0.00,0.99,0.00,0.17,0.00,0.00,0.03
claude-opus-4-8,"{'comma': 8, 'period': 3, 'straight_apos': 2}","{'comma': 7, 'period': 3, 'colon': 1, 'straigh...","{'comma': 14, 'period': 6, 'em_dash': 1, 'hyph...","{'comma': 18, 'period': 7, 'semicolon': 2, 'co...","{'comma': 9, 'period': 6, 'semicolon': 1, 'col...","{'comma': 20, 'period': 8, 'semicolon': 2, 'co...","{'comma': 5, 'period': 2, 'colon': 1, 'em_dash...","{'comma': 8, 'period': 3, 'em_dash': 1}","{'comma': 23, 'period': 7, 'em_dash': 3, 'hyph...","{'comma': 8, 'period': 5, 'colon': 1, 'em_dash...",...,0.00,0.00,0.59,0.00,0.91,0.00,0.13,0.01,0.00,0.04
claude-opus-5,"{'comma': 6, 'period': 3, 'hyphen': 1, 'straig...","{'comma': 19, 'period': 13, 'semicolon': 2, 'c...","{'comma': 3, 'period': 3, 'semicolon': 1, 'em_...","{'comma': 6, 'period': 6, 'semicolon': 1, 'col...","{'comma': 8, 'period': 7, 'colon': 2, 'em_dash...","{'comma': 10, 'period': 6, 'semicolon': 1, 'co...","{'comma': 9, 'period': 6, 'semicolon': 1, 'col...","{'comma': 17, 'period': 10, 'semicolon': 1, 'e...","{'comma': 14, 'period': 5, 'em_dash': 1, 'spac...","{'comma': 7, 'period': 3, 'colon': 1, 'hyphen'...",...,0.30,0.01,0.61,0.00,1.02,0.00,0.21,0.01,0.02,0.05
claude-sonnet-5,"{'comma': 12, 'period': 2, 'em_dash': 1, 'stra...","{'comma': 18, 'period': 6, 'colon': 1, 'em_das...","{'comma': 12, 'period': 5, 'colon': 1, 'hyphen...","{'comma': 14, 'period': 6, 'semicolon': 1, 'co...","{'comma': 4, 'period': 4, 'semicolon': 1, 'str...","{'comma': 15, 'period': 6, 'semicolon': 1, 'em...","{'comma': 16, 'period': 5, 'semicolon': 1, 'co...","{'comma': 25, 'period': 10, 'colon': 1, 'em_da...","{'comma': 7, 'period': 4, 'colon': 1, 'em_dash...","{'comma': 5, 'period': 2, 'colon': 1, 'em_dash...",...,0.00,0.00,0.89,0.00,1.52,0.00,0.24,0.00,0.01,0.01
gemini-3.1-pro-preview,"{'comma': 9, 'period': 3}","{'comma': 11, 'period': 6}","{'comma': 7, 'period': 5, 'semicolon': 1}","{'comma': 18, 'period': 12}","{'comma': 4, 'period': 4}","{'comma': 17, 'period': 10}","{'comma': 7, 'period': 5, 'colon': 1}","{'comma': 4, 'period': 3}","{'comma': 15, 'period': 6}","{'comma': 7, 'period': 6, 'colon': 1, 'hyphen'...",...,0.00,0.00,0.12,0.05,0.13,0.00,0.03,0.00,0.01,0.05
gemini-3.6-flash,"{'comma': 12, 'period': 4}","{'comma': 9, 'period': 5, 'curly_apos': 1}","{'comma': 7, 'period': 4, 'straight_apos': 1}","{'comma': 5, 'period': 4, 'semicolon': 1}","{'comma': 2, 'period': 3}","{'comma': 10, 'period': 4, 'semicolon': 1}","{'comma': 13, 'period': 8, 'colon': 1, 'hyphen...","{'comma': 11, 'period': 6}","{'comma': 7, 'period': 4, 'straight_apos': 1}","{'comma': 2, 'period': 3, 'hyphen': 1}",...,0.00,0.00,0.14,0.21,0.26,0.00,0.0

In [16]:
# lowercase words for each passage
train['words'] = train.text.str.lower().apply(lambda t: re.findall(r"[a-z]+(?:['’][a-z]+)?", t))

def overused(m, n=15, min_prompts=10):
    mine = train[train.model == m]
    rest = train[train.model != m]

    c_mine = Counter(w for ws in mine.words for w in ws)
    c_rest = Counter(w for ws in rest.words for w in ws)
    n_mine, n_rest = sum(c_mine.values()), sum(c_rest.values())

    # how many different prompts this model used each word in
    prompts_used = Counter(w for ws in mine.words.apply(set) for w in ws)

    # log-ratio: how much more often this model uses the word than everyone else
    score = {w: np.log((c_mine[w] + 1) / n_mine) - np.log((c_rest[w] + 1) / n_rest)
             for w in c_mine if prompts_used[w] >= min_prompts}

    return sorted(score, key=score.get, reverse=True)[:n]

overused_words = pd.DataFrame({m: overused(m) for m in sorted(train.model.unique())}).T
overused_words.to_csv("overused_words.csv", index=False)

overused_words

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
claude-fable-5,complete,everything,nearly,dissolves,nothing,moment,know,perfect,impossible,most,ever,simply,perhaps,self,years
claude-opus-4-6,roughly,emerges,cathedral,molecular,been,civilization,possible,most,something,billion,mere,surface,nearly,always,entire
claude-opus-4-8,quietly,very,dissolves,person,sense,thousand,upon,here,since,nearly,you,patient,how,hands,suddenly
claude-opus-5,thing,hundred,ten,he,million,somewhere,arithmetic,four,arrives,certain,last,read,learned,thousand,holds
claude-sonnet-5,since,earth's,held,mere,entirely,just,near,structure,intelligence,themselves,heartbeat,minds,some,self,though
gemini-3.1-pro-preview,ultimate,profound,chaotic,jagged,heavy,brilliant,biological,violently,relentless,exact,completely,absolute,microscopic,fleeting,sudden
gemini-3.6-flash,dynamic,breathtaking,expanse,vibrant,terrifying,realm,total,subtle,internal,delicate,violent,global,phantom,ultimately,fundamental
gemini-3.7-flash,fundamentally,perpetual,expanse,celestial,primordial,radiant,boundless,profound,intricate,sacred,completely,artificial,biological,colossal,relentless
gemini-3.8-flash,enduring,colossal,transforming,abyss,instantly,blind,crushing,absolute,solitary,biological,suspended,void,architecture,proving,modern
glm-5.3,billion,instant,hundred,now,since,moment,way,everything,nothing,call,miles,old,itself,years,sun
